# 03 - FedAvg: inti Federated Learning (TANPA SFM)

**Tujuan belajar:** membangun FedAvg dari nol dan melihat ia mendekati centralized
TANPA memindahkan data -- hanya berbagi BOBOT model.

**Mekanik satu ronde FedAvg:**
1. Server broadcast bobot global `W_t` ke semua klien.
2. Tiap klien latih `W_t` di data lokalnya E epoch -> dapat `W_k`.
3. Server agregasi: `W_{t+1} = sum_k (n_k / N) * W_k` (rata-rata TERTIMBANG ukuran data).
4. Ulangi T ronde.

Simulasi semua klien di notebook ini (belajar). Di riset survey, langkah 1-3
dijalankan lintas EC2 via S3 (`aws/fed_node_survey.py`) -- logikanya IDENTIK.

**Prasyarat:** `01_data_partition.ipynb` (folder `fl_data/`). Dijalankan di SageMaker.


In [ ]:
import sys
!{sys.executable} -m pip install -q numpy scikit-learn
print('deps OK')

## 1. MLP + FedAvg + FedProx (identik fed_node_survey.py)

`train_local` sudah menyertakan proximal term FedProx (`mu`); dengan `mu=0` ia jadi
FedAvg biasa. Kita pakai `mu=0` di notebook ini, dan `mu>0` dibahas di nb05.


In [ ]:
import numpy as np
SEED=42
def init_weights(d, seed=42):
    r=np.random.RandomState(seed); L=[d,128,64,1]; W=[]
    for a,b in zip(L[:-1],L[1:]):
        W.append(r.randn(a,b)*np.sqrt(2.0/a)); W.append(np.zeros(b))
    return W
def _relu(x): return np.maximum(0,x)
def _sig(x):  return 1.0/(1.0+np.exp(-np.clip(x,-30,30)))
def forward(W,X):
    a1=X@W[0]+W[1]; z1=_relu(a1)
    a2=z1@W[2]+W[3]; z2=_relu(a2)
    a3=z2@W[4]+W[5]; out=_sig(a3).ravel()
    return out,(X,a1,z1,a2,z2,a3,out)
def train_local(W,X,y,epochs=1,lr=3e-3,batch=256,mu=0.0,W_global=None,seed=42):
    r=np.random.RandomState(seed); n=len(y); W=[w.copy() for w in W]
    for _ in range(epochs):
        idx=r.permutation(n)
        for s in range(0,n,batch):
            bi=idx[s:s+batch]; Xb=X[bi]; yb=y[bi].astype(float)
            out,(X_,a1,z1,a2,z2,a3,o)=forward(W,Xb)
            g=(o-yb)/len(yb)
            gW4=z2.T@g[:,None]; gb4=g.sum(0)
            d2=(g[:,None]@W[4].T)*(a2>0); gW2=z1.T@d2; gb2=d2.sum(0)
            d1=(d2@W[2].T)*(a1>0);        gW0=X_.T@d1; gb0=d1.sum(0)
            for i,gv in enumerate([gW0,gb0,gW2,gb2,gW4,gb4]):
                gv=np.asarray(gv).reshape(W[i].shape)
                if mu>0 and W_global is not None: gv=gv+mu*(W[i]-W_global[i])
                W[i]=W[i]-lr*gv
    return W
def fedavg(weight_list, sizes):
    tot=float(sum(sizes)); out=[]
    for i in range(len(weight_list[0])):
        out.append(sum((sizes[k]/tot)*weight_list[k][i] for k in range(len(weight_list))))
    return out
def predict(W,X): return forward(W,X)[0]
def zscore_fit(X): mu=X.mean(0); sd=X.std(0); sd[sd==0]=1.0; return mu,sd
def zscore_apply(X,mu,sd): return (X-mu)/sd
print('FedAvg/FedProx siap (identik node survey)')

## 2. Muat partisi + z-score LOKAL per-klien

Penting: tiap klien fit z-score dari datanya SENDIRI (privasi). Statistik tidak dibagi.


In [ ]:
import os
DATA='./fl_data'
assert os.path.isdir(DATA), 'Jalankan 01_data_partition.ipynb dulu'
clients=sorted(set(int(f.split('_')[1]) for f in os.listdir(DATA) if f.startswith('client_')))
cli={}
for i in clients:
    X=np.load(f'{DATA}/client_{i}_train.npz')['X']; y=np.load(f'{DATA}/client_{i}_train.npz')['y']
    mu,sd=zscore_fit(X); cli[i]=dict(Xz=zscore_apply(X,mu,sd), y=y, mu=mu, sd=sd, n=len(y))
g=np.load(f'{DATA}/global_test.npz'); Xg,yg=g['X'],g['y']
d=Xg.shape[1]
print('klien:', clients, '| d =', d, '| total train =', sum(cli[i]['n'] for i in clients))

## 3. Loop FedAvg

Catatan: setiap klien pakai z-score lokalnya sendiri, jadi untuk mengevaluasi model
GLOBAL di test bersama kita harus memilih satu transform. Di sini kita evaluasi dengan
z-score tiap klien lalu rata-rata skornya (konsisten dengan realita deployment di mana
tiap node menormalkan inputnya sendiri).


In [ ]:
from sklearn.metrics import matthews_corrcoef, f1_score
def eval_global(W):
    ms=[]
    for i in clients:
        p=predict(W, zscore_apply(Xg, cli[i]['mu'], cli[i]['sd']))
        yp=(p>=0.5).astype(int); ms.append(matthews_corrcoef(yg,yp))
    return float(np.mean(ms))

ROUNDS=30; LOCAL_EPOCHS=2; MU=0.0   # mu=0 -> FedAvg murni
W=init_weights(d, SEED)
hist=[]
for t in range(ROUNDS):
    cw=[]; cs=[]
    for i in clients:
        Wl=train_local(W, cli[i]['Xz'], cli[i]['y'], epochs=LOCAL_EPOCHS, lr=3e-3, mu=MU, W_global=W, seed=SEED)
        cw.append(Wl); cs.append(cli[i]['n'])
    W=fedavg(cw,cs)
    if t%5==0 or t==ROUNDS-1:
        m=eval_global(W); hist.append((t,m)); print(f'ronde {t:2d}: MCC global = {m:.4f}')
fed_mcc=eval_global(W)
print('\nFedAvg final MCC (global):', round(fed_mcc,4))

## 4. Bandingkan dengan baseline nb02

Jika `learn_out/02_baselines.csv` ada, kita tempatkan FedAvg di antara
lokal-saja (bawah) dan centralized (atas).


In [ ]:
import csv
os.makedirs('learn_out',exist_ok=True)
base={}
p='learn_out/02_baselines.csv'
if os.path.exists(p):
    for row in csv.DictReader(open(p)): base[row['method']]=float(row['mcc'])
    print('centralized  MCC:', base.get('centralized'))
    print('FedAvg       MCC:', round(fed_mcc,4))
    print('lokal-saja   MCC:', base.get('local_only_mean'))
else:
    print('(jalankan nb02 dulu untuk pembanding penuh)')
with open('learn_out/03_fedavg.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['round','mcc_global']); [w.writerow([t,round(m,4)]) for t,m in hist]
    w.writerow(['final',round(fed_mcc,4)])
print('tersimpan: learn_out/03_fedavg.csv')
print('Langkah berikut: 04_noniid_dirichlet.ipynb -- apa yang terjadi saat klien TIDAK IID.')